In [5]:
import json
import pandas as pd

rows = []

with open("geschaefte.jsonl", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

states= rows[2]["drafts"][0]['states']
state_list = []
for state in states:
    state_list.append(state["id"])
if 222 in state_list:
    success=1
    
    

In [2]:
rows[2]

{'id': 20253002,
 'updated': '2025-11-14T03:32:41Z',
 'additionalIndexing': '15;08',
 'affairType': {'abbreviation': 'Po.', 'id': 6, 'name': 'Postulat'},
 'author': {'committee': {'abbreviation': 'APK-NR',
   'id': 4,
   'name': 'Aussenpolitische Kommission NR',
   'abbreviation1': 'APK-N',
   'abbreviation2': 'APK',
   'committeeNumber': 4,
   'council': {'abbreviation': 'NR',
    'id': 1,
    'name': 'Nationalrat',
    'type': 'N'},
   'typeCode': 1},
  'type': 'author'},
 'deposit': {'council': {'abbreviation': 'NR',
   'id': 1,
   'name': 'Nationalrat',
   'type': 'N'},
  'date': '2025-01-13T00:00:00Z',
  'legislativePeriod': 52,
  'session': '5207'},
 'descriptors': [],
 'drafts': [{'consultation': {'resolutions': [{'category': {'id': 3,
       'name': 'Normal'},
      'council': {'abbreviation': 'NR',
       'id': 1,
       'name': 'Nationalrat',
       'type': 'N'},
      'date': '2025-03-20T00:00:00Z',
      'text': 'Annahme',
      'type': 20}]},
   'federalCouncilProposal': {

In [6]:
df_Int = (
    df[df["type"] == "Interpellation"]
  .groupby(["person_code", "author"])
    .size()
)

NameError: name 'df' is not defined

In [7]:
flat = []

for g in rows:

    if "councillor" not in g.get("author", {}):
        continue

    states = g["drafts"][0]["states"]

    state_list = []
    for state in states:
        state_list.append(state["id"])

    if 209 in state_list:
        success = 1
    else:
        success = 0

    flat.append({
        "person_code": g["author"]["councillor"]["code"],
        "author": g["author"]["councillor"]["name"],
        "type": g["affairType"]["name"],
        "success": success
    })

df = pd.DataFrame(flat)
df["success"].value_counts()

success
0    4174
1     282
Name: count, dtype: int64

In [26]:
df_Int = (
    df[df["type"] == "Interpellation"]
    .groupby(["person_code", "author"])
    .size()
)

df_Post = (
    df[df["type"] == "Postulat"]
    .groupby(["person_code", "author"])
    .size()
)

df_Mot = (
    df[df["type"] == "Motion"]
    .groupby(["person_code", "author"])
    .size()
)

df_tot = (
    df.groupby(["person_code", "author"])
    .size()
)

df_success_int = (
    df[df["type"] == "Interpellation"]
    .groupby(["person_code", "author"])["success"]
    .sum()
)

df_success_post = (
    df[df["type"] == "Postulat"]
    .groupby(["person_code", "author"])["success"]
    .sum()
)

df_success_mot = (
    df[df["type"] == "Motion"]
    .groupby(["person_code", "author"])["success"]
    .sum()
)

df_success_total = (
    df.groupby(["person_code", "author"])["success"]
    .sum()
)

df_stats = pd.DataFrame({
    "total": df_tot,

    "motionen": df_Mot,
    "postulate": df_Post,
    "interpellationen": df_Int,

    "success_total": df_success_total,
    "success_motionen": df_success_mot,
    "success_postulate": df_success_post,
    "success_interpellationen": df_success_int
}).fillna(0).astype(int)

df_stats["mot_and_post"] = df_stats["motionen"] + df_stats["postulate"]
df_stats["success_mot_and_post"] = (
    df_stats["success_motionen"] + df_stats["success_postulate"]
)

In [27]:
df_stats = df_stats.reset_index()

In [28]:
df_stats.sort_values("success_total", ascending=False).head(100)

,person_code,author,total,motionen,postulate,interpellationen,success_total,success_motionen,success_postulate,success_interpellationen,mot_and_post,success_mot_and_post
190,10805,Broulis Pascal,42,12,6,24,11,7,4,0,18,11
124,3153,Friedli Esther,21,15,3,3,9,6,3,0,18,9
33,2776,Regazzi Fabio,33,11,5,17,8,4,4,0,16,8
110,3136,Würth Benedikt,18,11,3,4,7,4,3,0,14,7
32,2775,Caroni Andrea,10,5,3,2,6,3,3,0,8,6
...,...,...,...,...,...,...,...,...,...,...,...,...
31,2771,Müller-Altermatt Stefan,21,6,3,12,1,1,0,0,9,1
222,10838,Vontobel Erich,20,7,0,13,1,1,0,0,7,1
38,2796,Amaudruz Céline,23,17,2,3,1,1,0,0,19,1
36,2786,Grossen Jürg,31,12,1,18,1,1,0,0,13,1


In [32]:
df_stats[df_stats["author"] == "Page Pierre-André"]

,person_code,author,total,motionen,postulate,interpellationen,success_total,success_motionen,success_postulate,success_interpellationen,mot_and_post,success_mot_and_post
80,3090,Page Pierre-André,12,3,0,9,2,2,0,0,3,2


In [30]:
df[
    (df["author"] == "Badran Jacqueline") &
    (df["type"] == "Motion")
]

,person_code,author,type,success
453,2762,Badran Jacqueline,Motion,0
2145,2762,Badran Jacqueline,Motion,1
2183,2762,Badran Jacqueline,Motion,0
2803,2762,Badran Jacqueline,Motion,0
3732,2762,Badran Jacqueline,Motion,0


In [31]:
import json

liste = df_stats.to_dict(orient="records")

with open("parlamentarier_stats.json", "w", encoding="utf-8") as f:
    json.dump(liste, f, ensure_ascii=False, indent=2)